# BeltWatch: train on Kaggle

Builds the dataset and runs **one** job per run, chosen by `MODEL` below. See `docs/training.md`.

- `MODEL = "unet"` or `"segformer"` trains that neural model and evaluates it on val.
- `MODEL = "baselines"` needs no GPU. It re-scores attached neural checkpoints on val and compares them (paired bootstrap). It also evaluates the all-background baseline and tunes the random forest on val (`configs/random_forest_tuning.yaml`).
- `MODEL = "analysis"` needs no GPU. For each attached checkpoint it fits a temperature on the calibration split (fitted on all pixels, and separately on foreground pixels), checks both temperatures on val, writes an error-analysis report on val, and benchmarks the model on the session's CPU.

**Before the first run**, in the notebook's right-hand panel (Session options):
- **Accelerator:** GPU T4 x2 for `unet` and `segformer`. Avoid the P100: recent CUDA builds of PyTorch may not support it. For `baselines` and `analysis`, choose **None** (CPU), which uses no GPU quota.
- **Internet:** on. Kaggle needs a phone-verified account for this.

**For `baselines` and `analysis`:** upload the trained runs as a **private** Kaggle Dataset (Datasets → New Dataset). Each run is a folder named after it (`unet-resnet18-ce`, `segformer-b0-ce`) holding `best.pt` and `run.json`; a zip of those folders also works. Then attach it with **Add Input → Your Datasets**. The SegFormer weights are for noncommercial research only, so keep the dataset private.

**To run:** set `MODEL`, then **Save Version → Save & Run All (Commit)**. The run continues in the background without an open browser tab and saves `/kaggle/working/beltwatch-runs` as the version's output. A training run takes about 5 to 6 hours of the 12-hour limit: roughly 1 hour of data preparation, then up to 4.5 hours of training. A `baselines` run takes about 1 hour of data preparation plus a few hours on the CPU; an `analysis` run, about 1 hour plus 1 to 2 hours.

**If a run stops early:** open the notebook, choose **Add Input → Your Work → this notebook**, select the stopped version's output, and commit again. The notebook copies that output back. Training resumes from `latest.pt`; the forest tuning reuses the forests and reports it already saved.

In [ ]:
import os
import shutil

MODEL = "unet"  # "unet", "segformer" (GPU), "baselines" or "analysis" (CPU): one per session

WORK = "/kaggle/working/beltwatch-runs"  # saved as the version's output (20 GB limit)
MODELS = {
    "unet": ("configs/unet.yaml", "unet-resnet18-ce"),
    "segformer": ("configs/segformer_b0.yaml", "segformer-b0-ce"),
}
if MODEL not in (*MODELS, "baselines", "analysis"):
    raise ValueError(f"unknown MODEL {MODEL!r}")
GPU = MODEL in MODELS


def scratch_root() -> str:
    # The dataset (about 16 GB with extraction) stays off /kaggle/working, which is published
    # as output. Use whichever local disk has the most space.
    candidates = [p for p in ("/kaggle/temp", "/tmp", "/root") if os.path.isdir(p)]
    return max(candidates, key=lambda p: shutil.disk_usage(p).free)


REPO = os.path.join(scratch_root(), "beltwatch")
os.makedirs(WORK, exist_ok=True)
print("model:", MODEL, "| outputs:", WORK, "| repo:", REPO)
!df -h /kaggle/working {os.path.dirname(REPO)}

## Restore a previous run's output (only when resuming)

In [ ]:
def find_previous_output() -> str | None:
    # Kaggle mounts an attached notebook output somewhere under /kaggle/input.
    for root, dirs, _ in os.walk("/kaggle/input"):
        if root.count(os.sep) > 5:
            dirs.clear()
        elif os.path.basename(root) == "beltwatch-runs":
            return root
    return None


previous = find_previous_output()
if previous and not os.listdir(WORK):
    shutil.copytree(previous, WORK, dirs_exist_ok=True)
    print("restored", previous)
else:
    print("no previous output attached" if not previous else "outputs already present")

## Set up

In [ ]:
if os.path.isdir(REPO):
    !git -C {REPO} pull --ff-only
else:
    !git clone https://github.com/professor3333/beltwatch.git {REPO}
%cd {REPO}
!git log -1 --oneline
!curl -LsSf https://astral.sh/uv/install.sh | sh
os.environ["PATH"] = "/root/.local/bin:" + os.environ["PATH"]
TORCH_EXTRA = "cu126" if GPU else "cpu"
!uv sync --extra {TORCH_EXTRA}
if GPU:
    !uv run python -c "import torch; ok = torch.cuda.is_available(); print('CUDA:', ok, torch.cuda.get_device_name(0) if ok else ''); assert ok, 'no GPU: set Accelerator to GPU T4 x2'"

## Build the dataset
Downloads and verifies the pinned 7.5 GB archive (resumable), then validates, audits duplicates and assigns splits. The dataset-version files are copied into the output so they can be committed through a PR.

If the rebuilt files match the commit except for the download time in `zerowaste-f-download.json` (and the `dvc.lock` hash that follows from it), the committed copies are restored. Runs are then recorded against a clean commit instead of `-dirty`. Any other difference is reported and left in place.

In [ ]:
import json
import subprocess

!uv run dvc repro
!mkdir -p {WORK}/dataset-version
!cp -r data/manifests data/splits dvc.lock {WORK}/dataset-version/
!grep -m1 split_manifest_id data/splits/zerowaste-f-splits.json


def restore_committed_dataset_version() -> None:
    record = "data/manifests/zerowaste-f-download.json"
    status = subprocess.run(
        ["git", "status", "--porcelain", "--untracked-files=no"],
        capture_output=True,
        text=True,
        check=True,
    ).stdout
    changed = sorted(line[3:] for line in status.splitlines() if line)
    if not changed:
        print("dataset-version files match the commit")
        return

    def without_time(text: str) -> dict:
        return {k: v for k, v in json.loads(text).items() if k != "created_at"}

    committed = subprocess.run(
        ["git", "show", f"HEAD:{record}"], capture_output=True, text=True, check=True
    ).stdout
    with open(record) as fh:
        same_record = without_time(fh.read()) == without_time(committed)
    if set(changed) <= {record, "dvc.lock"} and same_record:
        subprocess.run(["git", "checkout", "--", *changed], check=True)
        print("only the download time differed; restored the committed", ", ".join(changed))
    else:
        print("WARNING: the rebuilt dataset differs from the commit:", ", ".join(changed))
        print("Runs will be recorded as -dirty. Commit the files in dataset-version/ through a PR.")


restore_committed_dataset_version()

## Train and evaluate on val (`unet`, `segformer`)

In [ ]:
import yaml

if GPU:
    BASE_CONFIG, RUN_DIR_NAME = MODELS[MODEL]
    RUN_DIR = f"{WORK}/models/{RUN_DIR_NAME}"
    with open(BASE_CONFIG) as fh:
        config = yaml.safe_load(fh)
    config["output_dir"] = RUN_DIR
    config["tracking"]["mlflow_uri"] = f"sqlite:///{WORK}/mlflow.db"
    config["runtime"]["num_workers"] = min(4, os.cpu_count() or 1)
    with open("/kaggle/working/train-config.yaml", "w") as fh:
        yaml.safe_dump(config, fh)

    RESUME = f"--resume {RUN_DIR}/latest.pt" if os.path.exists(f"{RUN_DIR}/latest.pt") else ""
    print("resume:", RESUME or "no, starting fresh")
    !uv run python scripts/train.py --config /kaggle/working/train-config.yaml {RESUME}
    !uv run python scripts/evaluate.py --model {MODEL} --model-dir {RUN_DIR} --split val --out-dir {WORK}/reports

## Baselines and paired comparisons on val (`baselines`)
Re-scores every attached checkpoint on val, so each report gets the per-image confusions that the paired bootstrap needs, and compares the neural models. Then it evaluates the all-background baseline, tunes the random forest on val, and compares each neural model with the tuned forest. The test split is never read.

In [ ]:
import glob
import zipfile


def find_checkpoints() -> dict[str, str]:
    """Run directory name -> path, for every attached folder with best.pt and run.json."""

    def scan() -> dict[str, str]:
        found = {}
        for root, _, files in os.walk("/kaggle/input"):
            if {"best.pt", "run.json"} <= set(files):
                found.setdefault(os.path.basename(root), root)
        for root, _, files in os.walk(EXTRACTED):
            if {"best.pt", "run.json"} <= set(files):
                found.setdefault(os.path.basename(root), root)
        return found

    found = scan()
    if not found:  # an attached zip that Kaggle did not unpack
        for archive in glob.glob("/kaggle/input/**/*.zip", recursive=True):
            with zipfile.ZipFile(archive) as zf:
                zf.extractall(EXTRACTED)
        found = scan()
    return found


def model_kind(run_name: str) -> str | None:
    return next((kind for kind in MODELS if run_name.startswith(kind)), None)


def report_dirs(names: list[str]) -> list[str]:
    return [
        d
        for d in (f"{WORK}/reports/{n}-val" for n in names)
        if os.path.exists(f"{d}/confusions.npz")
    ]


if not GPU:
    EXTRACTED = os.path.join(scratch_root(), "checkpoints")
    os.makedirs(EXTRACTED, exist_ok=True)
    checkpoints = {name: path for name, path in find_checkpoints().items() if model_kind(name)}
    print("checkpoints:", checkpoints or "none attached")

if MODEL == "baselines":
    for name, path in sorted(checkpoints.items()):
        args = f"--model {model_kind(name)} --model-dir {path} --split val --out-dir {WORK}/reports"
        !uv run python scripts/evaluate.py {args}
    neural = report_dirs(["segformer_b0", "unet_resnet18"])
    if len(neural) == 2:
        !uv run python scripts/compare.py {neural[0]} {neural[1]}

    !uv run python scripts/evaluate.py --model all-background --split val --out-dir {WORK}/reports
    !uv run python -m beltwatch.baselines.tune_random_forest --work-dir {WORK}/models/random-forest-tuning --out {WORK}/models/random-forest --reports-dir {WORK}/reports
    forest = report_dirs(["random-forest"])
    assert forest, "random-forest tuning did not finish; see the log above"
    for report in neural:
        pair = f"{report} {forest[0]}"
        !uv run python scripts/compare.py {pair}
    print(sorted(os.listdir(f"{WORK}/reports")))

## Calibration, error analysis and CPU benchmark (`analysis`)
For each attached checkpoint:

1. Fits a temperature on the **calibration split only**, once on all pixels and once on foreground pixels, with reliability diagrams.
2. Re-scores val with each temperature, so calibrated ECE is checked on data the temperature was not fitted on. Temperature does not change the predicted class, so IoU and coverage stay the same.
3. Writes the error-analysis report on val (uncalibrated; the errors do not depend on the temperature).
4. Builds an unactivated release bundle at temperature 1 and benchmarks it with 4 threads on 20 val frames. The bundle exists only to run the benchmark and is not a release candidate.

The test split is never read.

In [ ]:
if MODEL == "analysis":
    assert checkpoints, "attach the trained runs as a Kaggle Dataset (see the first cell)"
    VAL_FRAMES = "data/raw/zerowaste-f/splits_final_deblurred/val/data"
    BENCH_RELEASES = f"{WORK}/benchmark-releases"
    for name, path in sorted(checkpoints.items()):
        kind = model_kind(name)
        for scope in ("all", "foreground"):
            cal = f"{WORK}/calibration/{name}-{scope}"
            !uv run python -m beltwatch.evaluation.calibration --model {kind} --model-dir {path} --fit-scope {scope} --out {cal}
            with open(f"{cal}/calibration.json") as fh:
                temperature = json.load(fh)["temperature"]
            args = f"--model {kind} --model-dir {path} --split val --temperature {temperature}"
            !uv run python scripts/evaluate.py {args} --out-dir {WORK}/reports/calibrated-{scope}
        !uv run python scripts/error_report.py --model {kind} --model-dir {path} --out {WORK}/error-reports/{name}
        version = f"benchmark-{name}"
        if not os.path.exists(f"{BENCH_RELEASES}/{version}"):  # bundles are immutable
            note = "benchmark only; not a release candidate"
            !uv run python -m beltwatch.release.bundle --kind {kind} --model {path}/best.pt --version {version} --releases-dir {BENCH_RELEASES} --note "{note}"
        !uv run python scripts/benchmark.py --release {BENCH_RELEASES}/{version} --images {VAL_FRAMES} --threads 4 --out {WORK}/benchmarks/{name}.json
    !nproc
    !free -g
    print(sorted(os.listdir(WORK)))

## Afterwards
Download the version's output (`beltwatch-runs/`).

- **After training:** bring back `mlflow.db`, `reports/`, `dataset-version/`, and `models/<run>/best.pt` with `run.json`. `latest.pt` is only needed to resume.
- **After `baselines`:** bring back `reports/` (every report with its `confusions.npz`, and the `compare-*.json` files), `models/random-forest/tuning.json` and `metadata.json`, and `models/random-forest-tuning/reports/`. The forest itself (`model.joblib`) is only needed to rerun it.
- **After `analysis`:** bring back `calibration/`, `reports/calibrated-all/` and `reports/calibrated-foreground/`, `error-reports/`, and `benchmarks/`. The bundles in `benchmark-releases/` hold the weights again and are not needed. The error reports contain ZeroWaste frames: keep them out of git.